# 🎓 KHÓA LUẬN TỐT NGHIỆP: HUẤN LUYỆN MÔ HÌNH PHOBERT PHÂN LOẠI Ý ĐỊNH CHO HỆ THỐNG ERP AETHERPC
**Mô hình nền tảng:** `vinai/phobert-base-v2`  
**Mục tiêu:** Fine-tuning mô hình ngôn ngữ tiếng Việt để phân loại ý định (Intent Classification) người dùng cho trợ lý ảo ERP, sau đó xuất ra mô hình ONNX để nhúng vào Backend Node.js.

In [ ]:
# Bước 1: Cài đặt các thư viện cần thiết
!pip install -q transformers pyvi scikit-learn seaborn onnx onnxruntime accelerate

In [ ]:
# Bước 2: Kiểm tra GPU Google Colab
import torch
print(f"CUDA có sẵn: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Bước 3: Nạp dataset_intent.json trực tiếp
import json
import os
import pandas as pd

# Nếu chạy trên Colab độc lập, có thể đọc từ file upload hoặc nhúng trực tiếp
if os.path.exists('dataset_intent.json'):
    with open('dataset_intent.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
else:
    print("Vui lòng upload file dataset_intent.json lên thư mục Colab!")

df = pd.DataFrame(data)
print(f"Tổng số mẫu dữ liệu: {len(df)}")
print(df['intent'].value_counts())

In [ ]:
# Bước 4: Chạy huấn luyện (Fine-tune PhoBERT)
!python train_phobert.py

In [ ]:
# Bước 5: Hiển thị Biểu đồ Loss/Accuracy và Ma trận nhầm lẫn (Confusion Matrix)
from IPython.display import Image, display

if os.path.exists('model_checkpoint/training_curves.png'):
    display(Image('model_checkpoint/training_curves.png'))

if os.path.exists('model_checkpoint/confusion_matrix.png'):
    display(Image('model_checkpoint/confusion_matrix.png'))

if os.path.exists('model_checkpoint/classification_report.txt'):
    with open('model_checkpoint/classification_report.txt', 'r', encoding='utf-8') as f:
        print(f.read())

In [ ]:
# Bước 6: Xuất sang định dạng ONNX
!python export_onnx.py

In [ ]:
# Bước 7: Tải về các file kết quả và file mô hình ONNX để đưa vào hệ thống
import shutil
from google.colab import files

shutil.make_archive('phobert_kltn_artifacts', 'zip', 'model_checkpoint')
files.download('phobert_kltn_artifacts.zip')